MILESTONE 01: DATA COLLECTION & PREPARATION

Project: Unified Military Analytics & Comparison Dashboard

Objective: Scrape military data from GlobalFirepower.com, clean, and prepare analysis-ready dataset for KPI engineering and dashboard development.

SECTION 1: IMPORT LIBRARIES

Required libraries for web scraping, data processing, and cleaning
- requests: HTTP requests to fetch web pages
- BeautifulSoup: HTML parsing and data extraction
- pandas: Data manipulation and DataFrame operations
- numpy: Numerical operations
- re: Regular expressions for cleaning text
- time: Rate limiting to avoid blocking
- warnings: Suppress unnecessary warnings for cleaner output

In [7]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


SECTION 2: SCRAPING FUNCTIONS

These functions handle the core scraping logic:
1. scrape_main_ranking() - Extracts country names and Power Index from main page
2. scrape_metric() - Extracts individual metrics from detail pages
3. clean_metric() - Cleans messy data (removes $, commas, units, extracts numbers)

In [2]:
def scrape_main_ranking(url):
    """
    Scrape main ranking page to extract country names and Power Index scores.

    Parameters:
    -----------
    url : str
        URL of the main ranking page

    Returns:
    --------
    pandas.DataFrame
        DataFrame with columns: 'Country', 'Power_Index'
    """
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    countries_data = []
    records = soup.find_all('div', class_='recordsetContainer')

    for record in records:
        # Extract country name
        name_container = record.find('div', class_='longFormName')
        name = name_container.get_text(strip=True) if name_container else None

        # Extract Power Index (format: "PwrIndx: 0.0741")
        pi_container = record.find('div', class_='pwrIndxContainer')
        pwr_index = None
        if pi_container:
            pi_text = pi_container.get_text(separator=' ', strip=True).upper()
            if 'PWRINDX' in pi_text:
                try:
                    val_part = pi_text.split(':')[-1].strip()
                    pwr_index = float(val_part)
                except (ValueError, IndexError):
                    pass

        if name:
            countries_data.append({'Country': name, 'Power_Index': pwr_index})

    return pd.DataFrame(countries_data)


def scrape_metric(metric_name, url):
    """
    Scrape individual metric pages for each country.

    Parameters:
    -----------
    metric_name : str
        Name of the metric being scraped (will be column name)
    url : str
        URL of the metric page

    Returns:
    --------
    pandas.DataFrame
        DataFrame with columns: 'Country', {metric_name}
    """
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    metric_data = []
    records = soup.find_all('div', class_='recordsetContainer')

    for record in records:
        name_tag = record.find('div', class_='longFormName')
        if not name_tag:
            continue
        name = name_tag.get_text(strip=True)
        val_container = record.find('div', class_='valueContainer')
        val = val_container.get_text(strip=True) if val_container else None

        if name:
            metric_data.append({'Country': name, metric_name: val})

    return pd.DataFrame(metric_data)


def clean_metric(value):
    """
    Clean messy data by removing symbols, units, and extracting numbers.

    Handles cases like:
    - "$831,500,000,000" → 831500000000
    - "20,953,000 bbl" → 20953000
    - "Stock: 274\nReadiness: 178" → 274
    - "N/A" → None

    Parameters:
    -----------
    value : any
        Raw value from scraping (could be string, number, or None)

    Returns:
    --------
    float or None
        Cleaned numeric value or None if no number found
    """
    if pd.isna(value) or value is None:
        return None
    if isinstance(value, (int, float)):
        return value

    text = str(value)
    # Remove $, commas, tabs, newlines
    cleaned = re.sub(r'[$\\t\\r\\n,]', '', text)
    # Remove units (bbl, mt, km, sq km, etc.)
    cleaned = re.sub(r'(?i)(sq km|mt|bbl|ft|mi|km|bbl/day)', '', cleaned).strip()
    # Extract first valid number (integer or decimal)
    match = re.search(r'-?\d*\.?\d+', cleaned)
    return match.group(0) if match else None


print("Scraping functions defined successfully!")

Scraping functions defined successfully!


SECTION 3: KEY MAPPING FOR METRICS

This dictionary maps our desired column names to the exact text labels found on the GlobalFirepower website.
*  The keys are our standardized column names (used
in final dataset).

*   The values are the exact text strings from the website (case-sensitive).

In [3]:

key_map = {
    # Demographics & Economics
    "total_population": "Total Population:",
    "gdp": "Purchasing Power Parity:",
    "defense_budget": "Defense Budget:",
    "external_debt": "External Debt:",

    # Manpower
    "total_manpower": "Available Manpower",
    "active_personnel": "Active Personnel",
    "reserve_personnel": "Reserve Personnel",
    "air_force_personnel": "Air Force Personnel*",
    "army_personnel": "Army Personnel*",
    "navy_personnel": "Navy Personnel*",

    # Air Force
    "total_aircraft": "Aircraft Total:",
    "attack_aircraft": "Attack Types:",
    "fighter_aircraft": "Fighters:",
    "transport_aircraft": "Transports (Fixed-Wing):",
    "helicopters": "Helicopters:",
    "special_mission_aircraft": "Special-Mission:",
    "trainer_aircraft": "Trainers:",
    "attack_helicopters": "Attack Helicopters:",
    "tanker_aircraft": "Tanker Fleet:",

    # Naval Assets
    "naval_assets": "Total Assets:",
    "aircraft_carriers": "Aircraft Carriers:",
    "helicopter_carriers": "Helicopter Carriers:",
    "destroyers": "Destroyers:",
    "frigates": "Frigates:",
    "corvettes": "Corvettes:",
    "submarines": "Submarines:",
    "patrol_vessel": "Patrol Vessels:",
    "mine_warfare": "Mine Warfare:",

    # Land Forces
    "tanks": "Tanks:",
    "self_propelled_artillery": "Self-Propelled Artillery:",
    "towed_artillery": "Towed Artillery:",
    "rocket_artillery": "MLRS (Rocket Artillery):"
}

print(f"\n Key mapping created with {len(key_map)} metrics")
print(f"   Metrics: {list(key_map.keys())[:10]}...")


 Key mapping created with 32 metrics
   Metrics: ['total_population', 'gdp', 'defense_budget', 'external_debt', 'total_manpower', 'active_personnel', 'reserve_personnel', 'air_force_personnel', 'army_personnel', 'navy_personnel']...


SECTION 4: AUTOMATION FUNCTION

This function iterates through all country IDs, scrapes their detail pages,and extracts the metrics defined in key_map.

 Process:
1. For each country ID, construct the detail page URL
2. Parse HTML to find Power Index and all metrics
3. Extract values using the key_map labels
4. Store in dictionary and return as DataFrame

In [4]:
def automation(ids, key_map):
    """
    Scrape military data for all countries from GlobalFirepower.

    Parameters:
    -----------
    ids : list
        List of country IDs (e.g., ['united-states', 'russia'])
    key_map : dict
        Mapping of column names to website label text

    Returns:
    --------
    dict
        Dictionary with country names as keys and metric dicts as values
    """
    base_url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id="
    all_countries = {}

    for cid in ids:
        try:
            url = base_url + cid
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            # Extract country name from ID (convert hyphens to spaces, title case)
            country_name = cid.replace("-", " ").title()

            # Extract Power Index from the info block
            power_index = None
            info_block = soup.select_one("span.textNormal.textDkGray")
            if info_block:
                match = re.search(r"\d+\.\d+", info_block.text)
                if match:
                    power_index = match.group()

            # Extract all specifications from the specsGenContainers
            specs_containers = soup.find_all("div", class_="specsGenContainers")
            data_found = {}

            for container in specs_containers:
                tag = container.select_one("span.textLarge.textBold.textShadow")
                value_tags = container.select("span.textWhite.textShadow")

                if tag and value_tags:
                    name = tag.text.strip()
                    # Take the last value (GFP puts rank/label first, actual number last)
                    values = [v.text.strip() for v in value_tags]
                    data_found[name] = values[-1]

            # Map the found data to our standardized column names
            result = {}
            for new_key, old_key in key_map.items():
                if old_key in data_found:
                    result[new_key] = data_found[old_key]
                else:
                    result[new_key] = None

            # Add Power Index to the result
            result["power_index"] = power_index

            # Store in dictionary with country name as key
            all_countries[country_name] = result

            print(f"✅ Scraped: {country_name}")
            time.sleep(1)  # Rate limiting - be polite to the server

        except Exception as e:
            print(f"❌ Error scraping {cid}: {e}")

    return all_countries

print("\n Automation function defined successfully!")


 Automation function defined successfully!


SECTION 4.1: GET COUNTRY LIST FROM GLOBALFIREPOWER


In [8]:

print("\n" + "=" * 60)
print("STEP 1: EXTRACTING COUNTRY IDs")
print("=" * 60)

countries_url = "https://www.globalfirepower.com/countries-listing.php"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(countries_url, headers=headers, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

# Find all links and extract country IDs
country_tags = soup.find_all("a", href=True)
ids = []

for a in country_tags:
    href = str(a["href"])
    if "country-military-strength-detail.php?country_id=" in href:
        cid = href.split("country_id=")[1]
        ids.append(cid)

ids = list(set(ids))
print(f"✅ Found {len(ids)} unique country IDs")
print(f"   Sample IDs: {ids[:5]}")


STEP 1: EXTRACTING COUNTRY IDs
✅ Found 145 unique country IDs
   Sample IDs: ['uruguay', 'algeria', 'venezuela', 'lithuania', 'morocco']


SECTION 5: EXECUTE SCRAPING


*   Run the automation function to scrape all countries

*   This may take 2-3 minutes due to rate limiting

In [10]:
print("\n" + "=" * 60)
print("STEP 2: EXECUTING WEB SCRAPING")
print("=" * 60)
print("⏳ This may take 2-3 minutes for 145 countries...")
print("-" * 50)

all_countries_data = automation(ids, key_map)

# Convert to DataFrame
df = pd.DataFrame.from_dict(all_countries_data, orient="index")
df.reset_index(inplace=True)
df.rename(columns={"index": "country"}, inplace=True)

# CREATE DATA FOLDER IF IT DOESN'T EXIST
import os
if not os.path.exists('data'):
    os.makedirs('data')
    print("✅ Created 'data' folder")

# Save raw data immediately
df.to_csv("data/military_raw_data.csv", index=False)

print("\n" + "=" * 60)
print("SCRAPING COMPLETE!")
print("=" * 60)
print(f"✅ Total countries scraped: {len(df)}")
print(f"✅ Raw data saved: data/military_raw_data.csv")
print(f"📊 DataFrame shape: {df.shape}")


STEP 2: EXECUTING WEB SCRAPING
⏳ This may take 2-3 minutes for 145 countries...
--------------------------------------------------
✅ Scraped: Uruguay
✅ Scraped: Algeria
✅ Scraped: Venezuela
✅ Scraped: Lithuania
✅ Scraped: Morocco
✅ Scraped: Central African Republic
✅ Scraped: Afghanistan
✅ Scraped: Finland
✅ Scraped: Spain
✅ Scraped: Ethiopia
✅ Scraped: Japan
✅ Scraped: Burkina Faso
✅ Scraped: Vietnam
✅ Scraped: Kenya
✅ Scraped: Poland
✅ Scraped: United States Of America
✅ Scraped: Netherlands
✅ Scraped: Bolivia
✅ Scraped: Nepal
✅ Scraped: Ecuador
✅ Scraped: Armenia
✅ Scraped: Slovakia
✅ Scraped: Libya
✅ Scraped: Switzerland
✅ Scraped: Nicaragua
✅ Scraped: Zambia
✅ Scraped: Denmark
✅ Scraped: Suriname
✅ Scraped: Mauritania
✅ Scraped: Dominican Republic
✅ Scraped: Luxembourg
✅ Scraped: Turkmenistan
✅ Scraped: Australia
✅ Scraped: Guatemala
✅ Scraped: Honduras
✅ Scraped: India
✅ Scraped: Iran
✅ Scraped: United Kingdom
✅ Scraped: Lebanon
✅ Scraped: Iraq
✅ Scraped: Tanzania
✅ Scraped: Aus

SECTION 6: DATA CLEANING - STANDARDIZE COLUMN NAMES

*  Convert column names to snake_case for consistency
  
* Example: "total_population" instead of "Total Population"

In [11]:
print("\n" + "=" * 60)
print("STEP 3: DATA CLEANING - STANDARDIZING COLUMN NAMES")
print("=" * 60)

# Force UTF-8 output for proper text handling
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

# Standardize column names to snake_case
df.columns = (df.columns.str.lower()
              .str.strip()
              .str.replace(' ', '_')
              .str.replace('-', '_'))

print(f"Columns standardized: {list(df.columns[:10])}...")


STEP 3: DATA CLEANING - STANDARDIZING COLUMN NAMES
Columns standardized: ['country', 'total_population', 'gdp', 'defense_budget', 'external_debt', 'total_manpower', 'active_personnel', 'reserve_personnel', 'air_force_personnel', 'army_personnel']...


SECTION 7: DATA CLEANING - NUMERIC CONVERSION

Remove symbols ($ ,%,*,+) and covert to numeric values.

This function handles:

- Currency: "$831,500,000,000" → 831500000000
- Units: "20,953,000 bbl" → 20953000
- Readiness text: "Stock: 274" → 274
- Missing values: "N/A" → NaN

In [12]:
print("\n" + "=" * 60)
print("STEP 4: DATA CLEANING - NUMERIC CONVERSION")
print("=" * 60)

def clean_numeric_value(x):
    """
    Clean messy numeric strings and convert to float.

    Handles:
    - Currency symbols ($)
    - Commas (thousands separators)
    - Percent signs (%)
    - Plus signs (+)
    - Text like "Stock: 274" (extracts number)
    """
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    # Remove $, commas, %, + symbols
    x = x.replace(',', '').replace('$', '').replace('%', '').replace('+', '')

    # Check for N/A or null values
    if x.lower() in ['n/a', 'nan', 'null', '', 'none']:
        return np.nan

    # Extract first number using regex
    match = re.search(r'([\d.]+)', x)
    if match:
        return float(match.group(1))
    return np.nan

# Identify columns to clean (all except 'country')
cols_to_clean = [col for col in df.columns if col != 'country']
print(f"Cleaning {len(cols_to_clean)} columns...")

# Apply cleaning to each numeric column
for col in cols_to_clean:
    df[col] = df[col].apply(clean_numeric_value)

print(f"Cleaned {len(cols_to_clean)} numeric columns")


STEP 4: DATA CLEANING - NUMERIC CONVERSION
Cleaning 33 columns...
Cleaned 33 numeric columns


SECTION 8: HANDLING MISSING VALUES

Identify columns with missing data and fill appropriately
Strategy:
- For military asset columns (naval, aircraft, etc.), fill with 0(landlocked countries have 0 naval assets)
- For economic columns, fill with 0 (missing data means no data)

In [13]:
print("\n" + "=" * 60)
print("STEP 5: HANDLING MISSING VALUES")
print("=" * 60)

null_counts = df.isnull().sum()
missing_cols = null_counts[null_counts > 0]

if not missing_cols.empty:
    print("\n📊 Missing values found in:")
    for col, count in missing_cols.items():
        percentage = (count / len(df)) * 100
        print(f"   - {col}: {count} missing ({percentage:.1f}%)")

    # Fill missing values with 0
    df = df.fillna(0)
    print("\n✅ Applied rule: Filled missing values with 0")
else:
    print("\n✅ No missing values found")


STEP 5: HANDLING MISSING VALUES

📊 Missing values found in:
   - naval_assets: 32 missing (22.1%)
   - aircraft_carriers: 32 missing (22.1%)
   - helicopter_carriers: 32 missing (22.1%)
   - destroyers: 32 missing (22.1%)
   - frigates: 32 missing (22.1%)
   - corvettes: 32 missing (22.1%)
   - submarines: 32 missing (22.1%)
   - patrol_vessel: 32 missing (22.1%)
   - mine_warfare: 32 missing (22.1%)

✅ Applied rule: Filled missing values with 0


SECTION 9: VALIDATE CLEANED DATA

Perform quality checks to ensure data is ready for analysis

Checks performed:
 1. Row count (should be 140+ countries)
 2. Data types (metrics should be numeric)
 3. Missing values (should be 0%)

In [14]:
print("\n" + "=" * 60)
print("STEP 6: VALIDATING CLEANED DATA")
print("=" * 60)

row_count = len(df)
print(f"\n📊 Row count: {row_count} countries")
if row_count >= 140:
    print(f"✅ Row count check passed ({row_count} ≥ 140)")
else:
    print(f"⚠️  WARNING: Row count is {row_count}, expected 140+")

# Check data types
non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric_cols) > 1:  # Only 'country' should be non-numeric
    print(f"⚠️  WARNING: Unexpected non-numeric columns: {non_numeric_cols}")
else:
    print("✅ Data type check passed (all metrics are numeric)")

# Check for remaining nulls
if df.isnull().sum().sum() == 0:
    print("✅ Null value check passed (0% missing)")
else:
    print(f"⚠️  WARNING: {df.isnull().sum().sum()} null values remaining")



STEP 6: VALIDATING CLEANED DATA

📊 Row count: 145 countries
✅ Row count check passed (145 ≥ 140)
✅ Data type check passed (all metrics are numeric)
✅ Null value check passed (0% missing)


SECTION 10: EXPORT CLEANED DATASET

Save the cleaned, numeric dataset for KPI engineering

In [15]:
print("\n" + "=" * 60)
print("STEP 7: EXPORTING CLEANED DATASET")
print("=" * 60)

# Create data folder if it doesn't exist
import os
if not os.path.exists('data'):
    os.makedirs('data')

# Save cleaned dataset
df.to_csv("data/military_cleaned.csv", index=False)

print(f"✅ Cleaned data saved: data/military_cleaned.csv")
print(f"📊 Final shape: {df.shape}")
print(f"📁 File size: {os.path.getsize('data/military_cleaned.csv') / 1024:.1f} KB")


STEP 7: EXPORTING CLEANED DATASET
✅ Cleaned data saved: data/military_cleaned.csv
📊 Final shape: (145, 34)
📁 File size: 31.1 KB


SECTION 11: PREVIEW RESULTS

Display sample of cleaned data for verification

In [16]:
print("\n" + "=" * 60)
print("DATA PREVIEW")
print("=" * 60)

print("\n📋 First 5 rows (cleaned data):")
display(df.head())

print("\n📊 Top 10 Countries by Power Index:")
top_10 = df.nsmallest(10, 'power_index')[['country', 'power_index', 'defense_budget', 'total_aircraft', 'tanks']]
display(top_10)

print("\n📊 Data Types Summary:")
print(df.dtypes.value_counts())

print("\n" + "=" * 60)
print("MILESTONE 01 COMPLETE!")
print("=" * 60)
print("\n📌 Next: Milestone 02 - KPI Engineering")
print("   Input: military_cleaned.csv")
print("   Output: military_final.csv with KPIs and metadata")


DATA PREVIEW

📋 First 5 rows (cleaned data):


,country,total_population,gdp,defense_budget,external_debt,total_manpower,active_personnel,reserve_personnel,air_force_personnel,army_personnel,...,frigates,corvettes,submarines,patrol_vessel,mine_warfare,tanks,self_propelled_artillery,towed_artillery,rocket_artillery,power_index
0,Uruguay,3425330.0,1.085020e+11,1.903440e+09,5.445000e+10,1575652.0,24000.0,0.0,2850.0,15450.0,...,0.0,0.0,0.0,5.0,2.0,62.0,16.0,83.0,4.0,2.1537
1,Algeria,47022473.0,7.229120e+11,2.500000e+10,4.764000e+09,22570787.0,130000.0,150000.0,14000.0,280000.0,...,8.0,8.0,6.0,75.0,3.0,1485.0,224.0,483.0,188.0,0.4849
2,Venezuela,31250306.0,1.109430e+11,6.060000e+09,1.213630e+11,15625153.0,109000.0,8000.0,10755.0,300745.0,...,1.0,0.0,3.0,26.0,0.0,282.0,60.0,92.0,56.0,0.9106
3,Lithuania,2628186.0,1.362270e+11,5.651760e+09,4.840000e+10,1708321.0,23000.0,104000.0,1500.0,21000.0,...,0.0,0.0,0.0,5.0,4.0,0.0,16.0,54.0,0.0,1.8949
4,Morocco,37387585.0,3.505940e+11,1.610000e+10,4.226200e+10,17946041.0,400000.0,250000.0,13000.0,500000.0,...,6.0,1.0,0.0,84.0,0.0,1346.0,544.0,188.0,185.0,1.0368



📊 Top 10 Countries by Power Index:


,country,power_index,defense_budget,total_aircraft,tanks
15,United States Of America,0.0741,8.315000e+11,13032.0,4666.0
109,Russia,0.0791,2.126383e+11,4237.0,5630.0
44,China,0.0919,3.030000e+11,3529.0,5870.0
35,India,0.1346,1.090000e+11,2183.0,3913.0
102,South Korea,0.1642,4.480000e+10,1540.0,1831.0
121,France,0.1798,6.723240e+10,974.0,427.0
10,Japan,0.1876,5.800000e+10,1429.0,734.0
37,United Kingdom,0.1881,8.852653e+10,625.0,288.0
71,Turkey,0.1975,5.140000e+10,1101.0,2284.0
122,Italy,0.2211,3.732516e+10,714.0,203.0



📊 Data Types Summary:
float64    33
object      1
Name: count, dtype: int64

MILESTONE 01 COMPLETE!

📌 Next: Milestone 02 - KPI Engineering
   Input: military_cleaned.csv
   Output: military_final.csv with KPIs and metadata
